# ML Flow


In [2]:
import mlflow

from sklearn.model_selection import KFold, GridSearchCV
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.pipeline import Pipeline
import pandas as pd
import numpy as np
import os
#os.chdir('G:/My Drive/1Practical Machine Learning/notes/Cases/Medical_Cost_Personal')

In [3]:
mlflow.set_tracking_uri("http://localhost:5000")

In [4]:
# Start an MLflow experiment (you can change the experiment name)
mlflow.set_experiment("Medical Insurance Charges - RFRegressor")

<Experiment: artifact_location='mlflow-artifacts:/4', creation_time=1767155014951, experiment_id='4', last_update_time=1767155014951, lifecycle_stage='active', name='Medical Insurance Charges - RFRegressor', tags={}>

In [5]:
mlflow.sklearn.autolog()

In [6]:
med=pd.read_csv("D:\Machine Learning\Datasets\Cases\Medical_Cost_Personal\insurance.csv")
X, y = med.drop('charges', axis=1), med['charges']

In [7]:
ohe = OneHotEncoder(drop='first', sparse_output=False).set_output(transform='pandas')
col_trnf = ColumnTransformer([('OHE', ohe, make_column_selector(dtype_include=object))], # type: ignore
                             remainder='passthrough',
                             verbose_feature_names_out=False)
col_trnf = col_trnf.set_output(transform='pandas')


In [8]:

from sklearn.ensemble import RandomForestRegressor


In [ ]:
rf = RandomForestRegressor()
pipe = Pipeline([('TRNS',col_trnf), ('RF', rf)])
# el.get_params()
params = {
    'RF__n_estimators': [100, 200, 300, 500],           # more trees = better, but slower
    'RF__max_depth': [8, 12, 15, 20, None],
    'RF__min_samples_split': [2, 5, 10],
    'RF__min_samples_leaf': [1, 2, 4],
    'RF__max_features': ['sqrt', 'log2', 0.33, 1.0],    # ← changed None → 1.0
    'RF__bootstrap': [True, False]
    # Removed 'criterion' – default 'squared_error' is optimal; others are slower or not better
}



kfold = KFold(n_splits=5, shuffle=True, random_state=25)

gcv = GridSearchCV(pipe, param_grid=params, cv=kfold, scoring='r2', verbose=3)
gcv.fit(X, y)





2025/12/31 11:56:44 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID 'ee3ce5e5b97644ae9a3b327592feacb7', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2025/12/31 11:56:45 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "d:\Anaconda\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflo

Fitting 5 folds for each of 1440 candidates, totalling 7200 fits
[CV 1/5] END RF__bootstrap=True, RF__max_depth=8, RF__max_features=sqrt, RF__min_samples_leaf=1, RF__min_samples_split=2, RF__n_estimators=100;, score=0.873 total time=   0.1s
[CV 2/5] END RF__bootstrap=True, RF__max_depth=8, RF__max_features=sqrt, RF__min_samples_leaf=1, RF__min_samples_split=2, RF__n_estimators=100;, score=0.810 total time=   0.1s
[CV 3/5] END RF__bootstrap=True, RF__max_depth=8, RF__max_features=sqrt, RF__min_samples_leaf=1, RF__min_samples_split=2, RF__n_estimators=100;, score=0.830 total time=   0.1s
[CV 4/5] END RF__bootstrap=True, RF__max_depth=8, RF__max_features=sqrt, RF__min_samples_leaf=1, RF__min_samples_split=2, RF__n_estimators=100;, score=0.783 total time=   0.1s
[CV 5/5] END RF__bootstrap=True, RF__max_depth=8, RF__max_features=sqrt, RF__min_samples_leaf=1, RF__min_samples_split=2, RF__n_estimators=100;, score=0.840 total time=   0.1s
[CV 1/5] END RF__bootstrap=True, RF__max_depth=8, RF__m

In [ ]:
print(gcv.best_params_)
print(gcv.best_score_)

{'DT__criterion': 'absolute_error', 'DT__max_depth': 7, 'DT__max_features': None, 'DT__min_samples_leaf': 10, 'DT__min_samples_split': 5}
0.8487827968898651


In [ ]:
best=gcv.best_estimator_